## word2vec на PyTorch

Как вы уже могли заметить, идея, лежащая в основе [word2vec](https://arxiv.org/pdf/1310.4546), достаточно общая. В данном задании вы реализуете его самостоятельно.

Дисклеймер: не стоит удивляться тому, что реализация от `gensim` (или аналоги) обучается быстрее и работает точнее. Она использует множество доработок и ускорений, а также достаточно эффективный код. Ваша задача добиться промежуточных результатов за разумное время.

P.s. Как ни странно, GPU в этом задании нам не потребуется.

__Requirements:__ if you're running locally, in the selected environment run the following command:

```pip install --upgrade nltk bokeh umap-learn```


In [45]:
!pip install --upgrade nltk bokeh umap-learn


Defaulting to user installation because normal site-packages is not writeable


In [46]:
import itertools
import random
import string
from collections import Counter
from itertools import chain

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import umap
from IPython.display import clear_output
from matplotlib import pyplot as plt
from nltk.tokenize import WordPunctTokenizer
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR
from tqdm.auto import tqdm as tqdma

In [47]:
# download the data:
!wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora.txt -nc
# alternative download link: https://yadi.sk/i/BPQrUu1NaTduEw

"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [48]:
import urllib.request

url = "https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1"
urllib.request.urlretrieve(url, "quora.txt")

import os
print(os.path.exists("quora.txt"))
print(os.path.getsize("quora.txt"))

KeyboardInterrupt: 

In [49]:
data = list(open("./quora.txt", encoding="utf-8"))
data[50]

"What TV shows or books help you read people's body language?\n"

Токенизация – первый шаг.
Тексты, с которыми мы работаем, включают в себя пунктуацию, смайлики и прочие нестандартные токены, так что простой `str.split` не подойдет.

Обратимся к `nltk` - библиотеку, нашла широкое применеие в области NLP.

In [50]:
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

['What', 'TV', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [51]:
data_tok = [
    tokenizer.tokenize(
        line.translate(str.maketrans("", "", string.punctuation)).lower()
    )
    for line in data
]
data_tok = [x for x in data_tok if len(x) >= 3]

Несколько проверок:

In [52]:
assert all(
    isinstance(row, (list, tuple)) for row in data_tok
), "please convert each line into a list of tokens (strings)"
assert all(
    all(isinstance(tok, str) for tok in row) for row in data_tok
), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all("a" <= x.lower() <= "z" for x in tok)
assert all(
    map(lambda l: not is_latin(l) or l.islower(), map(" ".join, data_tok))
), "please make sure to lowercase the data"

Ниже заданы константы ширины окна контекста и проведена предобработка для построения skip-gram модели.

In [53]:
min_count = 5
window_radius = 5

In [54]:
vocabulary_with_counter = Counter(chain.from_iterable(data_tok))

word_count_dict = dict()
for word, counter in vocabulary_with_counter.items():
    if counter >= min_count:
        word_count_dict[word] = counter

vocabulary = set(word_count_dict.keys())
del vocabulary_with_counter

In [55]:
word_to_index = {word: index for index, word in enumerate(vocabulary)}
index_to_word = {index: word for word, index in word_to_index.items()}

Пары `(слово, контекст)` на основе доступного датасета сгенерированы ниже.

In [56]:
context_pairs = []

for text in data_tok:
    for i, central_word in enumerate(text):
        context_indices = range(
            max(0, i - window_radius), min(i + window_radius, len(text))
        )
        for j in context_indices:
            if j == i:
                continue
            context_word = text[j]
            if central_word in vocabulary and context_word in vocabulary:
                context_pairs.append(
                    (word_to_index[central_word], word_to_index[context_word])
                )

print(f"Generated {len(context_pairs)} pairs of target and context words.")

Generated 16576482 pairs of target and context words.


#### Подзадача №1: subsampling
Для того, чтобы сгладить разницу в частоте встречаемсости слов, необходимо реализовать механизм subsampling'а.
Для этого вам необходимо реализовать функцию ниже.

Вероятность **исключить** слово из обучения (на фиксированном шаге) вычисляется как
$$
P_\text{drop}(w_i)=1 - \sqrt{\frac{t}{f(w_i)}},
$$
где $f(w_i)$ – нормированная частота встречаемости слова, а $t$ – заданный порог (threshold).

In [57]:
def subsample_frequent_words(word_count_dict, threshold=1e-5):
    if not word_count_dict:
        return {}

    total_count = sum(word_count_dict.values())
    if total_count <= 0:
        return {word: 1.0 for word in word_count_dict}

    keep_prob_dict = {}
    for word, count in word_count_dict.items():
        if count <= 0:
            keep_prob_dict[word] = 1.0
            continue

        frequency = count / total_count
        keep_prob = np.sqrt(threshold / frequency)
        keep_prob_dict[word] = min(1.0, float(keep_prob))

    return keep_prob_dict

#### Подзадача №2: negative sampling
Для более эффективного обучения необходимо не только предсказывать высокие вероятности для слов из контекста, но и предсказывать низкие для слов, не встреченных в контексте. Для этого вам необходимо вычислить вероятност использовать слово в качестве negative sample, реализовав функцию ниже.

В оригинальной статье предлагается оценивать вероятность слов выступать в качестве negative sample согласно распределению $P_n(w)$
$$
P_n(w) = \frac{U(w)^{3/4}}{Z},
$$

где $U(w)$ распределение слов по частоте (или, как его еще называют, по униграммам), а $Z$ – нормировочная константа, чтобы общая мера была равна $1$.

In [ ]:
def get_negative_sampling_prob(word_count_dict):
    if not word_count_dict:
        return {}

    adjusted_counts = {
        word: count ** 0.75 for word, count in word_count_dict.items()
    }

    normalizer = sum(adjusted_counts.values())
    if normalizer == 0:
        return {word: 0.0 for word in word_count_dict}

    negative_sampling_prob_dict = {
        word: adjusted_count / normalizer
        for word, adjusted_count in adjusted_counts.items()
    }
    return negative_sampling_prob_dict

Для удобства, преобразуем полученные словари в массивы (т.к. все слова все равно уже пронумерованы).

In [59]:
keep_prob_dict = subsample_frequent_words(word_count_dict)
assert keep_prob_dict.keys() == word_count_dict.keys()

In [60]:
negative_sampling_prob_dict = get_negative_sampling_prob(word_count_dict)
assert negative_sampling_prob_dict.keys() == negative_sampling_prob_dict.keys()
assert np.allclose(sum(negative_sampling_prob_dict.values()), 1)

In [61]:
keep_prob_array = np.array(
    [keep_prob_dict[index_to_word[idx]] for idx in range(len(word_to_index))]
)
negative_sampling_prob_array = np.array(
    [
        negative_sampling_prob_dict[index_to_word[idx]]
        for idx in range(len(word_to_index))
    ]
)

Если все прошло успешно, функция ниже поможет вам с генерацией подвыборок (батчей).

In [62]:
def generate_batch_with_neg_samples(
    context_pairs,
    batch_size,
    keep_prob_array,
    word_to_index,
    num_negatives,
    negative_sampling_prob_array,
):
    if isinstance(context_pairs, np.ndarray):
        context_pairs_array = context_pairs
    else:
        context_pairs_array = np.asarray(context_pairs, dtype=np.int64)

    n_pairs = context_pairs_array.shape[0]
    selected_chunks = []
    collected = 0

    while collected < batch_size:
        needed = batch_size - collected
        candidate_size = max(needed * 2, 1024)
        candidate_idx = np.random.randint(0, n_pairs, size=candidate_size)
        candidates = context_pairs_array[candidate_idx]

        keep_mask = np.random.rand(candidate_size) < keep_prob_array[candidates[:, 0]]
        kept = candidates[keep_mask]
        if kept.shape[0] == 0:
            continue

        taken = kept[:needed]
        selected_chunks.append(taken)
        collected += taken.shape[0]

    batch = np.vstack(selected_chunks)
    neg_samples = np.random.choice(
        len(negative_sampling_prob_array),
        size=(batch_size, num_negatives),
        p=negative_sampling_prob_array,
    )
    return batch, neg_samples

In [63]:
batch_size = 4
num_negatives = 15
batch, neg_samples = generate_batch_with_neg_samples(
    context_pairs,
    batch_size,
    keep_prob_array,
    word_to_index,
    num_negatives,
    negative_sampling_prob_array,
)

Наконец, время реализовать модель. Обращаем ваше внимание, использование линейных слоев (`nn.Linear`) далеко не всегда оправданно!

Напомним, что в случае negative sampling решается задача максимизации следующего функционала:

$$
\mathcal{L} = \log \sigma({\mathbf{v}'_{w_O}}^\top \mathbf{v}_{w_I}) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)} \left[ \log \sigma({-\mathbf{v}'_{w_i}}^\top \mathbf{v}_{w_I}) \right],
$$

где:
- $\mathbf{v}_{w_I}$ – вектор центрального слова $w_I$,
- $\mathbf{v}'_{w_O}$ – вектор слова из контекста $w_O$,
- $k$ – число negative samplesЮ,
- $P_n(w)$ – распределение negative samples, заданное выше,
- $\sigma$ – сигмоида.

In [64]:
class SkipGramModelWithNegSampling(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, center_words, pos_context_words, neg_context_words):
        center_vectors = self.center_embeddings(center_words)
        pos_context_vectors = self.context_embeddings(pos_context_words)
        neg_context_vectors = self.context_embeddings(neg_context_words)

        pos_scores = torch.sum(center_vectors * pos_context_vectors, dim=1)
        neg_scores = torch.sum(
            center_vectors.unsqueeze(1) * neg_context_vectors, dim=2
        )

        return pos_scores, neg_scores

In [65]:
device = torch.device("cpu")

In [66]:
vocab_size = len(word_to_index)
embedding_dim = 32
num_negatives = 15

model = SkipGramModelWithNegSampling(vocab_size, embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.05)
lr_scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=150)
criterion = nn.BCEWithLogitsLoss()

In [67]:
params_counter = 0
for weights in model.parameters():
    params_counter += weights.shape.numel()
assert params_counter == len(word_to_index) * embedding_dim * 2

In [68]:
def train_skipgram_with_neg_sampling(
    model,
    context_pairs,
    keep_prob_array,
    word_to_index,
    batch_size,
    num_negatives,
    negative_sampling_prob_array,
    steps,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    device=device,
):
    pos_labels = torch.ones(batch_size).to(device)
    neg_labels = torch.zeros(batch_size, num_negatives).to(device)
    context_pairs_np = np.asarray(context_pairs, dtype=np.int64)
    loss_history = []
    for step in tqdma(range(steps)):
        batch, neg_samples = generate_batch_with_neg_samples(
            context_pairs_np,
            batch_size,
            keep_prob_array,
            word_to_index,
            num_negatives,
            negative_sampling_prob_array,
        )
        center_words = torch.tensor(batch[:, 0], dtype=torch.long, device=device)
        pos_context_words = torch.tensor(
            batch[:, 1], dtype=torch.long, device=device
        )
        neg_context_words = torch.tensor(neg_samples, dtype=torch.long).to(device)

        optimizer.zero_grad()
        pos_scores, neg_scores = model(
            center_words, pos_context_words, neg_context_words
        )

        loss_pos = criterion(pos_scores, pos_labels)
        loss_neg = criterion(neg_scores, neg_labels)

        loss = loss_pos + loss_neg
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        lr_scheduler.step(loss_history[-1])

        if step % 100 == 0:
            print(
                f"Step {step}, Loss: {np.mean(loss_history[-100:])}, learning rate: {lr_scheduler._last_lr}"
            )

In [69]:
steps = 2500
batch_size = 512
train_skipgram_with_neg_sampling(
    model,
    context_pairs,
    keep_prob_array,
    word_to_index,
    batch_size,
    num_negatives,
    negative_sampling_prob_array,
    steps,
)

  0%|          | 0/2500 [00:00<?, ?it/s]

Step 0, Loss: 4.68295955657959, learning rate: [0.05]
Step 100, Loss: 3.3851256132125855, learning rate: [0.05]
Step 200, Loss: 2.4542213797569277, learning rate: [0.05]
Step 300, Loss: 2.1543338894844055, learning rate: [0.05]
Step 400, Loss: 2.0049155807495116, learning rate: [0.05]
Step 500, Loss: 1.945026422739029, learning rate: [0.05]
Step 600, Loss: 1.9346899509429931, learning rate: [0.05]
Step 700, Loss: 1.943328878879547, learning rate: [0.025]
Step 800, Loss: 1.874780865907669, learning rate: [0.025]
Step 900, Loss: 1.7874541115760803, learning rate: [0.025]
Step 1000, Loss: 1.7263959121704102, learning rate: [0.025]
Step 1100, Loss: 1.6910977864265442, learning rate: [0.025]
Step 1200, Loss: 1.6527119040489198, learning rate: [0.025]
Step 1300, Loss: 1.6290875458717347, learning rate: [0.025]
Step 1400, Loss: 1.5929225492477417, learning rate: [0.0125]
Step 1500, Loss: 1.5560299122333527, learning rate: [0.0125]
Step 1600, Loss: 1.5285134744644164, learning rate: [0.0125]
S

Наконец, используйте полученную матрицу весов в качестве матрицы в векторными представлениями слов. Рекомендуем использовать для сдачи матрицу, которая отвечала за слова из контекста (т.е. декодера).

In [70]:
_model_parameters = model.parameters()
embedding_matrix_center = next(
    _model_parameters
).detach()  # Assuming that first matrix was for central word
embedding_matrix_context = next(
    _model_parameters
).detach()  # Assuming that second matrix was for context word

In [71]:
def get_word_vector(word, embedding_matrix, word_to_index=word_to_index):
    return embedding_matrix[word_to_index[word]]

Простые проверки:

In [74]:
# после обучения, перед assert-ами
_model_parameters = model.parameters()
embedding_matrix_center = next(_model_parameters).detach()
embedding_matrix_context = next(_model_parameters).detach()

# часто даёт заметно лучше качество, чем только context
embedding_matrix_context = (embedding_matrix_center + embedding_matrix_context) / 2


In [75]:
similarity_1 = F.cosine_similarity(
    get_word_vector("iphone", embedding_matrix_context)[None, :],
    get_word_vector("apple", embedding_matrix_context)[None, :],
)
similarity_2 = F.cosine_similarity(
    get_word_vector("iphone", embedding_matrix_context)[None, :],
    get_word_vector("dell", embedding_matrix_context)[None, :],
)
assert similarity_1 > similarity_2

In [76]:
similarity_1 = F.cosine_similarity(
    get_word_vector("windows", embedding_matrix_context)[None, :],
    get_word_vector("laptop", embedding_matrix_context)[None, :],
)
similarity_2 = F.cosine_similarity(
    get_word_vector("windows", embedding_matrix_context)[None, :],
    get_word_vector("macbook", embedding_matrix_context)[None, :],
)
assert similarity_1 > similarity_2

Наконец, взглянем на ближайшие по косинусной мере слова. Функция реализована ниже.

In [77]:
def find_nearest(word, embedding_matrix, word_to_index=word_to_index, k=10):
    word_vector = get_word_vector(word, embedding_matrix)[None, :]
    dists = F.cosine_similarity(embedding_matrix, word_vector)
    index_sorted = torch.argsort(dists)
    top_k = index_sorted[-k:]
    return [(index_to_word[x], dists[x].item()) for x in top_k.numpy()]

In [78]:
find_nearest("python", embedding_matrix_context, k=10)

[('java', 0.5795143246650696),
 ('backward', 0.5823718309402466),
 ('electric', 0.5862730741500854),
 ('beginners', 0.5885779857635498),
 ('archaeology', 0.6112178564071655),
 ('learning', 0.6270902156829834),
 ('biceps', 0.6361536383628845),
 ('structures', 0.6371079683303833),
 ('tests', 0.63787442445755),
 ('python', 1.0000001192092896)]

Также вы можете визуально проверить, как представлены в латентном пространстве часто встречающиеся слова.

In [79]:
top_k = 5000
_top_words = sorted([x for x in word_count_dict.items()], key=lambda x: x[1])[
    -top_k - 100 : -100
]  # ignoring 100 most frequent words
top_words = [x[0] for x in _top_words]
del _top_words

In [ ]:
word_embeddings = torch.cat(
    [embedding_matrix_context[word_to_index[x]][None, :] for x in top_words], dim=0
).numpy()

In [82]:
import sys
!{sys.executable} -m pip install bokeh

  Using cached xyzservices-2025.11.0-py3-none-any.whl.metadata (4.3 kB)
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/6.4 MB ? eta -:--:--
   --- ------------------------------------ 0.5/6.4 MB 1.2 MB/s eta 0:00:05
   ---- ----------------------------------- 0.8/6.4 MB 1.1 MB/s eta 0:00:06
   ------ --------------------------------- 1.0/6.4 MB 1.1 MB/s eta 0:00:05
   -------- ------------------------------- 1.3/6.4 MB 1.2 MB/s eta 0:00:05
   --------- ------------------------------ 1.6/6.4 MB 1.2 MB/s eta 0:00:05
   ----------- ---------------------------- 1.8/6.4 MB 1.2 MB/s eta 0:00:04
   ----------- ---------------------------- 1.8/6.4 MB 1.2 MB/s eta 0:00:04
   ------------- -------------------------- 2.1/6.4 MB 1.1 MB/s eta 0:00:04
   -------------- ------------------------- 2.4/6.4 MB 1.0 MB/s eta 0:00:04
   -------------- -------------------


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [83]:
import bokeh.models as bm
import bokeh.plotting as pl
from bokeh.io import output_notebook

output_notebook()


def draw_vectors(
    x,
    y,
    radius=10,
    alpha=0.25,
    color="blue",
    width=600,
    height=400,
    show=True,
    **kwargs,
):
    """draws an interactive plot for data points with auxilirary info on hover"""
    if isinstance(color, str):
        color = [color] * len(x)
    data_source = bm.ColumnDataSource({"x": x, "y": y, "color": color, **kwargs})

    fig = pl.figure(active_scroll="wheel_zoom", width=width, height=height)
    fig.scatter("x", "y", size=radius, color="color", alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show:
        pl.show(fig)
    return fig

Loading BokehJS ...

In [84]:
embedding = umap.UMAP(n_neighbors=5).fit_transform(word_embeddings)

In [85]:
draw_vectors(embedding[:, 0], embedding[:, 1], token=top_words)

figure(id='p1011', ...)

Для сдачи задания необходимо загрузить функции `subsample_frequent_words` и `get_negative_sampling_prob`, а также сгенерировать файл для посылки ниже и приложить в соответствующую задачу. Успехов!

In [86]:
# do not change the code in the block below
# __________start of block__________
import os
import json

assert os.path.exists(
    "words_subset.txt"
), "Please, download `words_subset.txt` and place it in the working directory"

with open("words_subset.txt") as iofile:
    selected_words = iofile.read().split("\n")


def get_matrix_for_selected_words(selected_words, embedding_matrix, word_to_index):
    word_vectors = []
    for word in selected_words:
        index = word_to_index.get(word, None)
        vector = [0.0] * embedding_matrix.shape[1]
        if index is not None:
            vector = embedding_matrix[index].numpy().tolist()
        word_vectors.append(vector)
    return word_vectors


word_vectors = get_matrix_for_selected_words(
    selected_words, embedding_matrix_context, word_to_index
)

with open("submission_dict.json", "w") as iofile:
    json.dump(word_vectors, iofile)
print("File saved to `submission_dict.json`")
# __________end of block__________

File saved to `submission_dict.json`
